# Hands-on: Can we detect breast cancer?

**Dataset:** Breast Cancer Wisconsin, real medical data built into scikit-learn. 569 patients, 30 measurements of each tumor (size, texture, smoothness...).

**Goal:** predict whether a tumor is **cancerous (malignant)** or **not (benign)**. This is a **classification** problem.

1. Load the data
2. Split, then put all columns on the same scale
3. Logistic regression
4. k-Nearest Neighbors
5. Decision tree
6. Confusion matrix for the best model
7. See overfitting happen

## Step 1. Load the data

In this dataset `0` means malignant and `1` means benign. We flip it so that **1 = cancer**, which makes recall mean *"how many cancers did we catch?"*

In [ ]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix

data = load_breast_cancer(as_frame=True)
X = data.data
y = 1 - data.target          # 1 = cancer (malignant), 0 = benign

print(X.shape)
print(y.value_counts())

## Step 2. Split, then put all columns on the same scale

- `stratify=y` keeps the same share of cancer cases in train and test.
- Scaling matters for k-NN, which measures distances. We learn the scale from the **training data only**.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler().fit(X_train)     # learn the scale from TRAIN only
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

print('Train:', len(X_train), '  Test:', len(X_test))

## Step 3. Logistic regression

In [ ]:
logreg = LogisticRegression().fit(X_train, y_train)
pred = logreg.predict(X_test)
print('Accuracy:', round(accuracy_score(y_test, pred), 3))
print('Recall  :', round(recall_score(y_test, pred), 3))

## Step 4. k-Nearest Neighbors (k = 5)

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5).fit(X_train, y_train)
pred = knn.predict(X_test)
print('Accuracy:', round(accuracy_score(y_test, pred), 3))
print('Recall  :', round(recall_score(y_test, pred), 3))

## Step 5. Decision tree (max depth 3)

In [ ]:
tree = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train, y_train)
pred = tree.predict(X_test)
print('Accuracy:', round(accuracy_score(y_test, pred), 3))
print('Recall  :', round(recall_score(y_test, pred), 3))

## Step 6. Confusion matrix for the best model

Change `logreg` to whichever model had the best recall.

```
[[true negatives,  false positives],
 [false negatives, true positives ]]
```

**False negatives** are the missed cancers: the most dangerous mistake here.

In [ ]:
pred = logreg.predict(X_test)        # change to your best model
print(confusion_matrix(y_test, pred))

## Step 7. See overfitting happen

Compare the **train** score with the **test** score. A big gap means overfitting.

In [ ]:
for k in [1, 5, 15, 50]:
    knn = KNeighborsClassifier(n_neighbors=k).fit(X_train, y_train)
    print(f'k={k:2d}  train={knn.score(X_train, y_train):.3f}  test={knn.score(X_test, y_test):.3f}')

## Your tasks

1. Fill in the table:

| Model | Accuracy | Recall |
|---|---|---|
| Logistic regression | | |
| k-NN (k = 5) | | |
| Decision tree | | |

2. Which model missed the fewest cancers (highest recall)?
3. In step 7, which *k* has the biggest train–test gap? Is that overfitting or underfitting?
4. In step 5, change `max_depth=3` to `max_depth=None` and compare `tree.score(X_train, y_train)` with `tree.score(X_test, y_test)`. Does the tree overfit?